# Task 10: Inference & Model Serving

Building a FastAPI REST API for BERT predictions.

## FastAPI Server Setup

In [ ]:
# inference_server.py
"""
FastAPI server for BERT inference.
Run with: uvicorn inference_server:app --reload
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
import torch.nn as nn
from typing import List, Optional

app = FastAPI(title="BERT Inference API", version="1.0.0")


# Model definition (same as training)
class BERTClassifier(nn.Module):
    def __init__(self, vocab_size=30000, d_model=128, num_heads=4, num_layers=2, d_ff=256, num_labels=2):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.encoder_layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.pooler = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(d_model, num_labels)
        
    def forward(self, token_ids):
        x = self.embeddings(token_ids)
        for layer in self.encoder_layers:
            x = layer(x)
        pooled = torch.tanh(self.pooler(x[:, 0]))
        return self.classifier(self.dropout(pooled))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        
    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        return self.norm2(x + self.ffn(x))


# Load model (in production, load saved weights)
model = BERTClassifier()
model.eval()
print("Model loaded!")


# Request/Response models
class PredictRequest(BaseModel):
    token_ids: List[int]

class PredictResponse(BaseModel):
    prediction: int
    probabilities: List[float]


@app.get("/")
async def root():
    return {"message": "BERT Inference API", "status": "running"}


@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    try:
        # Convert to tensor
        tokens = torch.tensor([request.token_ids])
        
        # Inference
        with torch.no_grad():
            logits = model(tokens)
            probs = torch.softmax(logits, dim=-1)[0]
            pred = logits.argmax(dim=-1).item()
        
        return PredictResponse(
            prediction=pred,
            probabilities=probs.tolist()
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/health")
async def health():
    return {"status": "healthy"}

print("\nServer code saved! Run: uvicorn inference_server:app --reload")

## Client Usage

In [ ]:
# client.py
"""Client for testing the BERT API."""

import requests

# Test the API
API_URL = "http://localhost:8000/predict"

# Sample token IDs (in production, use a tokenizer)
sample_tokens = [101, 2054, 2003, 2019, 2890, 102]  # "Who is king?"

response = requests.post(API_URL, json={"token_ids": sample_tokens})
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

In [ ]:
# Batch inference for efficiency
class BERTPredictor:
    """Batch predictor with caching."""
    
    def __init__(self, model, device='cuda', batch_size=32):
        self.model = model.to(device)
        self.model.eval()
        self.device = device
        self.batch_size = batch_size
        
    @torch.no_grad()
    def predict(self, token_ids_list):
        """Predict on a list of token ID lists."""
        results = []
        
        for i in range(0, len(token_ids_list), self.batch_size):
            batch = token_ids_list[i:i+self.batch_size]
            
            # Pad to max length in batch
            max_len = max(len(ids) for ids in batch)
            padded = torch.tensor([ids + [0]*(max_len-len(ids)) for ids in batch])
            padded = padded.to(self.device)
            
            logits = self.model(padded)
            preds = logits.argmax(dim=-1).tolist()
            results.extend(preds)
        
        return results

print("Batch predictor defined!")

## Summary
- ✅ FastAPI REST endpoints
- ✅ Pydantic request/response models
- ✅ Batch inference for efficiency
- ✅ Health checks

## Next: [Task 11: Production Optimization](../task-11-production-optimization/overview.html)